In [ ]:
# helper functions
# eventually add survivor tools
# make single game sim and RUF more robust

import pandas as pd
import numpy as np
import random
from scipy.stats import norm

def get_bracket_order(num_teams):
    seeds = [1, 2] # start with base 2 team bracket

    #keep doubling number of seeds, and paring approprately
    #e.g. 1 vs 16 winner players 8vs 9 winner etc.
    while len(seeds) < num_teams:
        next_val = len(seeds) * 2 + 1
        new_seeds = []
        for s in seeds:
            new_seeds.append(s)
            new_seeds.append(next_val - s)
        seeds = new_seeds        
    return seeds

def pred_score_diff(fav, dog, pace=100):
    #basic power ratings now from KP or EM(per game, not 100 poss)
    return (fav - dog) * pace / 100

def ruf(dif, gp):
    #rating update function, model better next year, but UA article is fine for now
    #rating diffi=ß1(scorediffi-1-exp scorediffi-1)+ß2(score diffi-1-exp score diffi-1)*ln(game number)
    #LRegression the change in team rating on score diff vs expectation. Have interaction of score diff relative to expectation with gp
    #conservative coefficients for now, model next year
    return np.clip(0.15 * dif - 0.03 * dif * np.log(gp), -3, 3)

def sim_match(pred_diff, fav, dog):
    #future sample from normal distribution centered around mean score differential. Varies based on spread and tot, but ~11.2 stdev
    #not good at ML, but fine for sim. in future, adjust for high variance teams e.g. 3 point and TO heavy teams
    
    # use rng to get margin of victory
    ruv = random.random()
    z = norm.ppf(ruv)
    stdev = 11.2

    #win_prob = 1/(1+np.exp(-.163*pred_diff))  #too simple, prefer margin of victory
    raw_margin = pred_diff + (z * stdev)
    sim_score_diff = round(raw_margin)
    
    # if rounded to 0, adjust - if 0.000 choose
    if sim_score_diff == 0: 
        sim_score_diff = np.ceil(raw_margin) if raw_margin > 0 else np.floor(raw_margin)
        
        if sim_score_diff == 0: 
            sim_score_diff = 1 if pred_diff >= 0 else -1

    return int(sim_score_diff)

def sim_round(teams_df):
    winners = []
    
    for i in range(0, len(teams_df), 2): #iterate in pairs
        team_a = teams_df.iloc[i]
        team_b = teams_df.iloc[i+1]
        
        #sim game
        pred_diff = pred_score_diff(team_a['Rating'], team_b['Rating'], pace=team_a['AdjT'])
        result = sim_match(pred_diff, team_a['Team'], team_b['Team'])
        
        if result > 0:
            winners.append(team_a)
        else:
            winners.append(team_b)
            
    return pd.DataFrame(winners)

def sim_tourney(data):
    num_teams = len(data)
    bracket_order = get_bracket_order(num_teams)
    current_round_teams = data.loc[bracket_order]
    #sim all rounds
    while len(current_round_teams) > 1:
        current_round_teams = sim_round(current_round_teams)
    return current_round_teams.iloc[0]


In [48]:
path = r"C:\Users\Cooper\Downloads\cbb.xlsx"
data = pd.read_excel(path, usecols=["Seed", "Team", "AdjT", "EM","KP"], index_col="Seed")
print(data.head())

                Team  AdjT     EM     KP
Seed                                    
1               Duke  65.4  21.97  37.68
16        St. John's  69.5  15.75  26.93
8     Michigan State  66.3  16.63  28.77
9        Connecticut  64.7  17.46  28.67
3               Iowa  62.7  13.01  23.85


In [50]:
#init tracking
num_sims = 123123
bracket_order = get_bracket_order(len(data))
ordered_data = data.loc[bracket_order]
team_names = ordered_data['Team'].values
ratings = ordered_data['KP'].values.astype(float)
num_teams = len(team_names)
max_rounds = int(np.log2(num_teams))
win_tracking = np.zeros((num_teams, max_rounds), dtype=int)

for sim in range(num_sims):
    # track winners and resim
    active_indices = np.arange(num_teams)
    
    for r in range(max_rounds):
        # ID home/away teams
        idx_a = active_indices[0::2]
        idx_b = active_indices[1::2]
        
        # Vectorized sim to make faster
        pred_diffs = (ratings[idx_a] - ratings[idx_b]) 
        z_scores = norm.ppf(np.random.random(len(idx_a)))
        sim_margins = pred_diffs + (z_scores * 11.2)
        
        # If margin >= 0, A wins
        winners_mask = sim_margins >= 0
        
        # Update for the next round
        active_indices = np.where(winners_mask, idx_a, idx_b)
        win_tracking[active_indices, r] += 1

win_counts = pd.DataFrame(win_tracking, index=team_names, 
    columns=[f"Wins_{i+1}" for i in range(max_rounds)])

win_odds = (win_counts / num_sims) * 100
print(win_odds.sort_values(by=f"Wins_{max_rounds}", ascending=False).head(16))

                   Wins_1     Wins_2     Wins_3     Wins_4
Arizona         87.004865  65.862593  40.435175  24.828830
Michigan        83.582271  62.552894  43.484970  24.506388
Duke            83.029978  65.408575  37.537259  22.419044
Illinois        51.018088  39.603486  18.252479   7.702054
Houston         48.981912  37.260301  16.844944   6.978387
Purdue          83.627754  28.024821  11.268406   4.346873
Iowa State      64.122057  22.527879  11.140892   4.029304
Michigan State  50.615238  13.730172   3.926967   1.175248
Connecticut     49.384762  13.423974   3.814072   1.154130
Nebraska        60.219455  15.772033   3.783209   0.798389
Tennessee       35.877943   8.604404   3.124518   0.750469
Alabama         16.417729   6.314823   2.081658   0.462952
St. John's      16.970022   7.437278   1.778709   0.439398
Arkansas        12.995135   4.599466   1.046921   0.214420
Iowa            39.780545   7.364181   1.287331   0.174622
Texas           16.372246   1.513121   0.192490   0.0194